# MMIBC v2 - Colab quickstart

Run the cells **top to bottom**. Runtime > Change runtime type > GPU for anything beyond the smoke test.

What lives where (nothing is lost when the runtime resets):

| Location | Content |
|---|---|
| Google Drive `MyDrive/MMIBC/processed`, `splits`, `runs`, `figures`, `cache` | processed images, frozen splits and pair tables, checkpoints, predictions, metrics, figures, model weights |
| Colab local disk `/content` | raw downloads (large; re-downloaded after a reset) and a fast staged copy of the processed images |

The data audit **stops the run on a real problem** (leakage, label mismatch, duplicate images across splits, ...). Read its message instead of working around it.

In [ ]:
import os
REPO = "/content/MMIBC"
if not os.path.exists(REPO):
    !git clone -b revision/v2 https://github.com/dev-tyta/MMIBC.git {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}
!pip -q install -e ".[download]"

In [ ]:
import torch
print("CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available (CPU only)")
from mmibc.utils.paths import resolve_paths
paths = resolve_paths()          # mounts Google Drive and creates the folder layout
print(paths)

## 1. Smoke test (about a minute, CPU is fine)
Runs the **whole** pipeline on synthetic images with a tiny random backbone: prepare, audit, both task types, evaluation with controls, explanations, figures. If this fails, fix it before downloading anything.

In [ ]:
!mmibc smoke

## 2. Credentials
Add these as Colab secrets (key icon, left bar) and switch **Notebook access ON** for each one:

* `KAGGLE_API_TOKEN` - dataset downloads. Create it in your Kaggle account Settings with **Generate New Token** (what current `kagglehub` expects). The older pair `KAGGLE_USERNAME` + `KAGGLE_KEY` ("Create Legacy API Key") also works.
* `HF_TOKEN` - **only** for DINOv3 (gated: accept the licence on the model page first)

Colab secrets can only be read by notebook cells, **not** by the `!mmibc ...` commands below (separate processes). The next cell copies them into environment variables, which those commands inherit. Re-run it after adding or changing a secret. Values are never printed.

In [ ]:
from mmibc.data.download import export_colab_secrets
export_colab_secrets()   # prints one status line per secret, never the value
# Need at least one Kaggle credential ("exported"/"already set") before section 3. HF_TOKEN may stay missing unless you use DINOv3.

## 3. Data
`mmibc download` with no argument lists every registered dataset. The Kaggle slugs for VinDr come from the old code; the BUSI slug is an **unverified mirror** - `inspect` shows what actually arrived, and the audit checks it.

In [ ]:
!mmibc download

In [ ]:
!mmibc download vindr_images vindr_annotations busi

In [ ]:
!mmibc inspect vindr_annotations
!mmibc inspect busi

`bus_bra` downloads from Zenodo; `bus_uclm` from Kaggle (run `!mmibc download bus_uclm` then `!mmibc inspect bus_uclm` - its layout is unverified, so you may need a `source_manifest.csv`). `breast_lesions_usg` is **manual** (TCIA): place it under `<local>/raw/breast_lesions_usg/` (the command prints the exact path and what to check). A source listed as `data.us_external_source` is kept out of training entirely and used as an external test set.

`cmmd` (Kaggle) is a **mammography** set with biopsy-confirmed labels, not ultrasound: it can be downloaded and inspected, but nothing in the pipeline uses it yet.

In [ ]:
!mmibc download bus_bra
# !mmibc inspect bus_bra     # check the layout; if it is not class folders, write raw/bus_bra/source_manifest.csv (uid, raw_path, class_name[, mask_paths, patient_id])
OVER = "data.us_sources=[busi,bus_bra]"
POOL = "run_name=pool_busi_busbra"     # a different pool is a different experiment: give it its own run folder

## 4. Prepare and audit
`prepare` is resumable (re-run it after an interruption). `audit` prints every check - schema, leakage, near-duplicates across splits, label mapping, source-label confounding, pair integrity - and trains nothing.

In [ ]:
!mmibc prepare vindr -c vindr_multiview
!mmibc prepare us -c mammo_us_synthetic --set {OVER}

In [ ]:
!mmibc audit -c vindr_multiview
!mmibc audit -c mammo_us_synthetic --set {OVER}

## 5. Experiments
**A. Native pairs (CC and MLO of the same breast, VinDr-Mammo)** - validates the cross-attention fusion on genuine correspondence.

In [ ]:
!mmibc run -c vindr_multiview

**B. Synthetic mammogram + ultrasound pairs and their controls.** `label_matched` is the paper's setting. `random` and `shuffled_partner` are the controls a reviewer will ask for: if they score the same, the fusion is not using the pairing.

In [ ]:
for mode in ("label_matched", "random", "shuffled_partner"):
    !mmibc run -c mammo_us_synthetic --set task.pair_mode={mode} {OVER} {POOL}

## 6. Ablations on identical frozen test pairs
`ladder` = fusion variants, `backbones` = DINOv2 / DINOv2+registers / DINOv3 / CNNs, `pairing` = pairing tiers. Each prints one table with DeLong and McNemar tests against the reference run.

In [ ]:
!mmibc suite ladder
# !mmibc suite backbones     # DINOv3 rows need HF_TOKEN
# !mmibc suite pairing

## 7. Comparison with published results
`protocol_petrini` reproduces the labelling and split of the closest published baseline, so that comparison row is meaningful. Every other row is flagged *not comparable* - read the caveat column before quoting it.

In [ ]:
!mmibc run -c protocol_petrini
import glob
run = sorted(glob.glob(str(paths.runs / "protocol_petrini*")))[-1]
!mmibc literature {run} --dataset VinDr

## 8. Figures for the paper

In [ ]:
!mmibc figures --template lncs          # ROC, ladder, controls, calibration, explanation summary, data composition -> <root>/figures

In [ ]:
# Architecture and algorithm diagrams (TikZ). Pre-built copies are in paper/figures/pdf; rebuild only if you change them.
!apt-get -qq install -y texlive-latex-extra texlive-fonts-recommended poppler-utils > /dev/null
!mmibc figures --tex
from IPython.display import Image, display
display(Image("paper/figures/pdf/fig_architecture.png", width=900))

## 9. Real mammogram + ultrasound pairs (KAU-BCMD) - needs one manual step
The corpus reports 205 ultrasound cases that correspond to part of the mammogram cases, but the layout of the Kaggle mirror is not known. So: download, **inspect**, then write the pair manifest yourself.

In [ ]:
!mmibc download kau_bcmd
!mmibc inspect kau_bcmd

Write `pairs_manifest.csv` with columns `pair_id, case_id, path_a, path_b, label` (`path_a` = mammogram, `path_b` = ultrasound, paths relative to two root folders, `label` 0/1 from the corpus labels, **one row per patient pairing**). Downscale the images first (long side 672) - full-size PNGs are slow. Then evaluate a trained synthetic-pair model on real pairs (Tier 1b, evaluation only):

In [ ]:
# !mmibc evaluate -c mammo_us_synthetic --set task.pair_mode=label_matched {OVER} {POOL} \
#       task.real_pairs_csv=/content/kau/pairs_manifest.csv task.real_root_a=/content/kau task.real_root_b=/content/kau

## Troubleshooting
* **`audit` fails** - the message names the check and the rows; fix the data, do not raise the tolerance.
* **Out of GPU memory** - `--set training.batch_size=8 model.freeze=last_n`, or a smaller backbone (`model.backbone=dinov2_small`).
* **Runtime disconnected during training** - run the same command again; training resumes from the last atomic checkpoint, including the random-number state.
* **DINOv3 `401/403`** - accept the licence on the model page and set `HF_TOKEN`; the default backbone `dinov2_base` needs neither.
* **Drive quota** - raw downloads stay on local disk by default (`paths.raw_on_drive=false`); only processed images, splits, runs and weights go to Drive.